In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from google.colab import files
files.upload()

In [2]:
df=pd.read_csv('Cleaned_Churn_Data.csv')
df.head()

,city,gender,senior_citizen,partner,dependents,tenure_months,phone_service,multiple_lines,internet_service,online_security,...,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn_reason,churn_value
0,Los Angeles,Male,No,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Competitor made better offer,1
1,Los Angeles,Female,No,No,Yes,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Moved,1
2,Los Angeles,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,...,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,Moved,1
3,Los Angeles,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,...,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Moved,1
4,Los Angeles,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,...,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,Competitor had better devices,1


###Some Mandatory Checks

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   city               7043 non-null   object 
 1   gender             7043 non-null   object 
 2   senior_citizen     7043 non-null   object 
 3   partner            7043 non-null   object 
 4   dependents         7043 non-null   object 
 5   tenure_months      7043 non-null   int64  
 6   phone_service      7043 non-null   object 
 7   multiple_lines     7043 non-null   object 
 8   internet_service   7043 non-null   object 
 9   online_security    7043 non-null   object 
 10  online_backup      7043 non-null   object 
 11  device_protection  7043 non-null   object 
 12  tech_support       7043 non-null   object 
 13  streaming_tv       7043 non-null   object 
 14  streaming_movies   7043 non-null   object 
 15  contract           7043 non-null   object 
 16  paperless_billing  7043 

In [4]:
df.isna().sum()

,0
city,0
gender,0
senior_citizen,0
partner,0
dependents,0
tenure_months,0
phone_service,0
multiple_lines,0
internet_service,0
online_security,0


In [5]:
df.columns

Index(['city', 'gender', 'senior_citizen', 'partner', 'dependents',
       'tenure_months', 'phone_service', 'multiple_lines', 'internet_service',
       'online_security', 'online_backup', 'device_protection', 'tech_support',
       'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing',
       'payment_method', 'monthly_charges', 'total_charges', 'churn_reason',
       'churn_value'],
      dtype='object')

###Dividing columns baded on usage

In [6]:
drop = ['city', 'total_charges', 'churn_reason']
keep = [col for col in df.columns if col not in drop]
data = df[keep]
x = data.drop('churn_value', axis=1)
y = data['churn_value']

In [7]:
data.dtypes

,0
gender,object
senior_citizen,object
partner,object
dependents,object
tenure_months,int64
phone_service,object
multiple_lines,object
internet_service,object
online_security,object
online_backup,object


###checking the unique values in each columns for preprocessing

In [8]:
for col in x.select_dtypes(include='object').columns:
  print(col,x[col].unique())

gender ['Male' 'Female']
senior_citizen ['No' 'Yes']
partner ['No' 'Yes']
dependents ['No' 'Yes']
phone_service ['Yes' 'No']
multiple_lines ['No' 'Yes' 'No phone service']
internet_service ['DSL' 'Fiber optic' 'No']
online_security ['Yes' 'No' 'No internet service']
online_backup ['Yes' 'No' 'No internet service']
device_protection ['No' 'Yes' 'No internet service']
tech_support ['No' 'Yes' 'No internet service']
streaming_tv ['No' 'Yes' 'No internet service']
streaming_movies ['No' 'Yes' 'No internet service']
contract ['Month-to-month' 'Two year' 'One year']
paperless_billing ['Yes' 'No']
payment_method ['Mailed check' 'Electronic check' 'Bank transfer (automatic)'
 'Credit card (automatic)']


#ENCODING

###LABEL ENCODING
We will encode the binary columns with label encoding.
* **gender** have male and female values which will be encoded
* **binary** cols have yes and no which can be encoded to 1 and 0
* **to_clean** columns have all common 'no internet service' which obviously means 'No' thats why we will replace them with "no' and then extend them with binary cols and then encode them.

In [9]:
data['gender'] = data['gender'].map({'Male': 1, 'Female': 0})
binary=['senior_citizen','partner','dependents','phone_service','paperless_billing']
to_clean=['online_security', 'online_backup', 'device_protection','tech_support', 'streaming_tv', 'streaming_movies']
for col in to_clean:
  data[col]=data[col].replace('No internet service', 'No')
binary.extend(to_clean)
for col in binary:
  data[col]=data[col].map({'Yes':1,'No':0})

Feature Engineering for numeric cols\
We make a new col **tenure_period** which will be based on **tenure_months** as short ,medium and long according to the customers tenure.\
Throug this we can make more dummy columns which will overall improve model accracy.

In [10]:
data['tenure_period'] = pd.cut(data['tenure_months'],bins=[-1, 12, 36, 72],labels=['short', 'medium', 'long'])

###ONE HOT ENCODING
We will not do one hot encoding in this because we will do taht later in ml pipeline so that we dont have to redo it again when we provide new data to model seperately.

In [11]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   gender             7043 non-null   int64   
 1   senior_citizen     7043 non-null   int64   
 2   partner            7043 non-null   int64   
 3   dependents         7043 non-null   int64   
 4   tenure_months      7043 non-null   int64   
 5   phone_service      7043 non-null   int64   
 6   multiple_lines     7043 non-null   object  
 7   internet_service   7043 non-null   object  
 8   online_security    7043 non-null   int64   
 9   online_backup      7043 non-null   int64   
 10  device_protection  7043 non-null   int64   
 11  tech_support       7043 non-null   int64   
 12  streaming_tv       7043 non-null   int64   
 13  streaming_movies   7043 non-null   int64   
 14  contract           7043 non-null   object  
 15  paperless_billing  7043 non-null   int64   
 16  paymen

In [12]:
data.to_csv('Processed_Churn_Data.csv', index=False)
files.download('Processed_Churn_Data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>